In [0]:
!pip install kaggle

In [0]:
import os

os.environ["KAGGLE_USERNAME"] = dbutils.secrets.get("kaggle-scope", "kaggle_username")
os.environ["KAGGLE_KEY"] = dbutils.secrets.get("kaggle-scope", "kaggle_key")

print("KAGGLE_USERNAME set:", bool(os.environ.get("KAGGLE_USERNAME")))
print("KAGGLE_KEY set:", bool(os.environ.get("KAGGLE_KEY")))


In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.ecommerce
""")

In [0]:
spark.sql("""
CREATE VOLUME IF NOT EXISTS workspace.ecommerce.ecommerce_data
""")

In [0]:
%sh
cd /Volumes/workspace/ecommerce/ecommerce_data

kaggle datasets download -d mkechinov/ecommerce-behavior-data-from-multi-category-store


In [0]:
%sh
cd /Volumes/workspace/ecommerce/ecommerce_data
unzip -o ecommerce-behavior-data-from-multi-category-store.zip
ls -lh

In [0]:
%sh
cd /Volumes/workspace/ecommerce/ecommerce_data
rm -f ecommerce-behavior-data-from-multi-category-store.zip
ls -lh

In [0]:

%restart_python

In [0]:
df_n = spark.read.csv("/Volumes/workspace/ecommerce/ecommerce_data/2019-Nov.csv")

In [0]:
df_o = spark.read.csv("/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv")

In [0]:
df_oct = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv")
)

df_nov = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/workspace/ecommerce/ecommerce_data/2019-Nov.csv")
)


In [0]:
df_oct = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv")
)

In [0]:
print(f"October 2019 - Total Events: {df_oct.count():,}")
print(f"Noveber 2019 - Total Events: {df_nov.count():,}")

In [0]:
print("\n" + "="*60)
print("SAMPLE DATA (First 5 rows):")
print("="*60)
df_nov.show(5, truncate=False)

In [0]:
df_oct = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv")
)

df_nov = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/workspace/ecommerce/ecommerce_data/2019-Nov.csv")
)

In [0]:

(df_nov.write
  .format("delta")
  .mode("overwrite")
  .saveAsTable("events_table")
)


In [0]:
%sql

SELECT COUNT(*) FROM events_table;
DESCRIBE HISTORY events_table;


In [0]:
%sql

DESCRIBE EXTENDED events_table;


In [0]:
# %python
print("Spark version:", spark.version)

# Serverless-safe: use spark.conf instead of spark.sparkContext
for k in [
    "spark.databricks.clusterUsageTags.clusterName",
    "spark.databricks.workspaceUrl",
    "spark.databricks.clusterUsageTags.clusterId",
]:
    try:
        print(f"{k} =", spark.conf.get(k))
    except Exception:
        pass

# Optional: user (may vary by workspace permissions)
try:
    print("User:", dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get())
except Exception:
    print("User: unavailable")



In [0]:
# %python
from pyspark.sql import functions as F

df = (
    spark.range(0, 2_000_000)
    .withColumnRenamed("id", "user_id")
    .withColumn(
        "country",
        F.when((F.col("user_id") % 5) == 0, "UK")
         .when((F.col("user_id") % 5) == 1, "IN")
         .when((F.col("user_id") % 5) == 2, "US")
         .when((F.col("user_id") % 5) == 3, "DE")
         .otherwise("FR")
    )
    .withColumn("spend", (F.col("user_id") % 100) * F.lit(1.25))
)

df


In [0]:
display(df)

In [0]:
# %python
agg_df = (
    df.filter(F.col("spend") > 50)
      .groupBy("country")
      .agg(
          F.count("*").alias("events"),
          F.round(F.avg("spend"), 2).alias("avg_spend"),
          F.max("spend").alias("max_spend")
      )
      .orderBy(F.desc("events"))
)

agg_df


In [0]:
# %python
agg_df.explain(True)


In [0]:
display(agg_df)

In [0]:
# %python
df.createOrReplaceTempView("events")


In [0]:
%sql
EXPLAIN FORMATTED
SELECT
  country,
  COUNT(*) AS events,
  ROUND(AVG(spend), 2) AS avg_spend,
  MAX(spend) AS max_spend
FROM events
WHERE spend > 50
GROUP BY country
ORDER BY events DESC;


In [0]:
%sql
SELECT
  country,
  COUNT(*) AS events,
  ROUND(AVG(spend), 2) AS avg_spend,
  MAX(spend) AS max_spend
FROM events
WHERE spend > 50
GROUP BY country
ORDER BY events DESC;


In [0]:
%fs
ls /Volumes/workspace/ecommerce/ecommerce_data/


In [0]:
df_nov.show(5, truncate=False)

In [0]:
from pyspark.sql import functions as F

filtered_df = (
    df_nov.filter(F.col("event_type").isin("purchase", "cart"))
      .filter(F.col("price") > 0)
      .select("event_time","event_type","product_id","category_code","brand","price","user_id","user_session")
)

# Materialize the cache (first action)
filtered_df.count()

# Use it multiple times (subsequent actions reuse cache)
display(
    filtered_df.groupBy("category_code")
      .agg(F.sum("price").alias("revenue"), F.count("*").alias("events"))
      .orderBy(F.desc("revenue"))
)


In [0]:
filtered_df.createOrReplaceTempView("filtered_events")

# Trigger materialization
spark.table("filtered_events").count()

In [0]:
%sql
SELECT category_code, SUM(price) AS revenue
FROM filtered_events
GROUP BY category_code
ORDER BY revenue DESC;


In [0]:
# %python
from pyspark.sql.window import Window

w = Window.partitionBy("category_code").orderBy(F.desc("revenue"))

top_brand_per_category = (
    df_nov.filter(F.col("event_type") == "purchase")
      .groupBy("category_code", "brand")
      .agg(F.sum("price").alias("revenue"))
      .withColumn("rank", F.row_number().over(w))
      .filter(F.col("rank") <= 3)
      .orderBy("category_code", "rank")
)

display(top_brand_per_category)


In [0]:
%sql
SELECT
  brand,
  ROUND(SUM(price), 2) AS total_revenue,
  COUNT(*) AS purchase_events,
  COUNT(DISTINCT user_id) AS unique_users
FROM filtered_events
WHERE event_type = 'purchase'
  AND price > 0
GROUP BY brand
ORDER BY total_revenue DESC
LIMIT 10;


In [0]:
# %python
brand_funnel = (
    df_nov.filter(F.col("event_type").isin("view", "purchase"))
      .groupBy("brand")
      .agg(
          F.sum(F.when(F.col("event_type") == "view", 1).otherwise(0)).alias("views"),
          F.sum(F.when(F.col("event_type") == "purchase", 1).otherwise(0)).alias("purchases"),
          F.round(F.sum("price"), 2).alias("revenue")
      )
      .withColumn(
          "conversion_rate",
          F.when(F.col("views") > 0, F.round(F.col("purchases") / F.col("views"), 4))
           .otherwise(F.lit(0))
      )
      .orderBy(F.desc("conversion_rate"))
      .limit(10)
)

display(brand_funnel)
